# Feature Engineering and Modelling

## BCG X — PowerCo Churn Investigation

This notebook uses the final dataset provided for the prediction task and trains a Random Forest classifier to predict customer churn.

## 1. Import packages

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt

%matplotlib inline
sns.set(color_codes=True)

---
## 2. Load data

In [ ]:
df = pd.read_csv('./data_for_predictions.csv')
df.drop(columns=['Unnamed: 0'], inplace=True)
df.head()

,id,cons_12m,cons_gas_12m,cons_last_month,forecast_cons_12m,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,...,months_modif_prod,months_renewal,channel_MISSING,channel_ewpakwlliwisiwduibdlfmalxowmwpci,channel_foosdfpfkusacimwkcsosbicdxkicaua,channel_lmkebamcaaclubfxadlmueccxoimlema,channel_usilxuppasemubllopkaafesmlibmsdf,origin_up_kamkkxfxxuwbdslkwifmmcsiusiuosws,origin_up_ldkssxwpmemidmecebumciepifcamkci,origin_up_lxidpiddsbxsbosboudacockeimpuepw
0,24011ae4ebbe3035111d65fa7c15bc57,0.000000,4.739944,0.000000,0.000000,0.0,0.444045,0.114481,0.098142,40.606701,...,2,6,0,0,1,0,0,0,0,1
1,d29c2c54acc38ff3c0614d0a653813dd,3.668479,0.000000,0.000000,2.280920,0.0,1.237292,0.145711,0.000000,44.311378,...,76,4,1,0,0,0,0,1,0,0
2,764c75f661154dac3a6c254cd082ea7d,2.736397,0.000000,0.000000,1.689841,0.0,1.599009,0.165794,0.087899,44.311378,...,68,8,0,0,1,0,0,1,0,0
3,bba03439a292a1e166f80264c16191cb,3.200029,0.000000,0.000000,2.382089,0.0,1.318689,0.146694,0.000000,44.311378,...,69,9,0,0,0,1,0,1,0,0
4,149d57cf92fc41cf94415803a877cb4b,3.646011,0.000000,2.721811,2.650065,0.0,2.122969,0.116900,0.100015,40.606701,...,71,9,1,0,0,0,0,1,0,0


---
## 3. Modelling

We use a Random Forest classifier to predict the `churn` target variable.

In [ ]:
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

### Data sampling

The data is split into training and test samples so that predictions can be evaluated on observations that were not used to train the model. Following the supplied notebook template, a 75-25% train-test split is used.

In [ ]:
# Separate target variable from independent variables
y = df['churn']
X = df.drop(columns=['id', 'churn'])

print(X.shape)
print(y.shape)

(14606, 61)
(14606,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

X_train: (10954, 61)
y_train: (10954,)
X_test: (3652, 61)
y_test: (3652,)


### Model training

A Random Forest is an ensemble of decision trees. The classifier is trained on the training sample and then used to generate predictions for the unseen test sample.

In [ ]:
# Train the Random Forest classifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

### Evaluation

We evaluate the trained model on the test dataset.

In [ ]:
# Generate predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print('First 10 predictions:')
print(y_pred[:10])

First 10 predictions:
[0 0 0 0 0 0 0 0 0 0]


### Evaluation metrics

**Accuracy** measures the overall proportion of correct predictions.

**Precision** measures how many customers predicted as churners actually churned.

**Recall** measures how many of the actual churners were identified by the model. This is particularly relevant for churn because missed churners are important from a customer-retention perspective.

**F1-score** combines precision and recall into a single measure and is useful when both types of classification error matter.

**ROC-AUC** measures how well the model separates the two classes across different classification thresholds.

In [ ]:
accuracy = metrics.accuracy_score(y_test, y_pred)
precision = metrics.precision_score(y_test, y_pred, zero_division=0)
recall = metrics.recall_score(y_test, y_pred, zero_division=0)
f1 = metrics.f1_score(y_test, y_pred, zero_division=0)
roc_auc = metrics.roc_auc_score(y_test, y_pred_proba)

results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC'],
    'Score': [accuracy, precision, recall, f1, roc_auc]
})

display(results)

,Metric,Score
0,Accuracy,0.903067
1,Precision,0.714286
2,Recall,0.054645
3,F1-score,0.101523
4,ROC-AUC,0.664881


In [ ]:
# Confusion matrix
confusion_matrix = metrics.confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(confusion_matrix)

Confusion Matrix:
[[3278    8]
 [ 346   20]]


### Model performance assessment

The model's overall accuracy is high, but accuracy alone does not provide a sufficient assessment of churn prediction because the churn classes are imbalanced.

For this model, recall is particularly important because it shows the proportion of actual churners identified. The test-set results should therefore be considered using recall, precision, F1-score and ROC-AUC alongside accuracy.

With the supplied dataset and the Random Forest configuration used above, the model identifies only a small proportion of actual churners. Therefore, the model performance is **not satisfactory for reliably identifying churners**, despite the relatively high overall accuracy. This conclusion is based on the test-set evaluation metrics rather than accuracy alone.

### Summary

- Random Forest classifier trained on the supplied `data_for_predictions.csv` dataset.
- 75% of observations were used for training and 25% for testing, following the supplied template.
- Accuracy, precision, recall, F1-score and ROC-AUC were used to evaluate performance.
- Recall is especially important for assessing the model's ability to identify customers who actually churn.